<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# From counts to meaning: bag of words, TF-IDF and embeddings

**[From counts to meaning: bag of words, TF-IDF and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m08/from-counts-to-meaning/)** · Deep Learning and Transformers Explained · Module 8 (optional), lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute word counts, cosine similarity and TF-IDF by hand with NumPy, check them against scikit-learn to 4 decimals, see which options change the numbers, and measure the synonym problem in three representations.

| | |
|---|---|
| **Time** | About 30 minutes. Nothing here trains for longer than a few seconds. |
| **Needs** | An internet connection for the first cells (about 2 MB of tickets and a 3.9 MB file of word vectors). No account. |
| **You should know** | The bag of words, from [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/), and the dot product and cosine similarity of the mathematics course. |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | Python 3.12 with scikit-learn 1.9.1, NumPy 2.5 and PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have these packages preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/from-counts-to-meaning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M08-L01-from-counts-to-meaning/from-counts-to-meaning-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Five documents as counts

The lesson uses five made-up documents. `CountVectorizer` builds the vocabulary and counts every word; we give it the course's `tokenize`, so it cuts text exactly as the course does.

Run the cell.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["my parcel is late", "the package has not arrived", "refund my card",
        "the parcel arrived broken", "my refund is late"]
cv = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
C = cv.fit_transform(docs).toarray()
words = list(cv.get_feature_names_out())
print(words)
print(C, C.shape)

> **Check.** 5 rows (documents) and 12 columns (words, in alphabetical order). d1 has a 1 under "is", "late", "my" and "parcel".

## 2. The tickets are almost all zeros

> **Predict.** A ticket has 2,864 places, one per vocabulary word. How many of them are not 0 in an average training ticket: about 40, 400 or 1,400?

Run the cell.

In [ ]:
vocab = Vocab([r["text"] for r in train_rows])
X = bag_of_words(train_rows, vocab)
nonzero = (X > 0).sum(dim=1).float()
print(tuple(X.shape), "| mean non-zeros per ticket", round(nonzero.mean().item(), 1))
print("share of zeros:", round(1 - nonzero.mean().item() / X.shape[1], 4))
print("bytes per ticket: full", X.shape[1] * 4, "| sparse about", round(nonzero.mean().item() * (4 + 4)))

> **Check.** 43.3 non-zeros on average, 98.5% zeros: 11,456 bytes as a full vector, and about 350 bytes when you store only the non-zero numbers and their positions (4 + 4 bytes each).

## 3. Cosine similarity by hand

The cosine similarity of two vectors is their dot product divided by both lengths: x · y ÷ (‖x‖ ‖y‖).

> **Your turn.** Write `cosine(x, y)` with NumPy (`x @ y` is the dot product, `np.linalg.norm(x)` the length). Then compute the cosine of d1 and d5 (`C[0]`, `C[4]`) in `cos_15` and of d1 and d2 in `cos_12`.

In [ ]:
def cosine(x, y):
    ...

cos_15 = ...
cos_12 = ...

In [ ]:
expect('cos_15', cos_15, 0.75, tolerance=1e-06)

In [ ]:
expect('cos_12', cos_12, 0.0, tolerance=1e-09)

d1 "my parcel is late" and d5 "my refund is late" share three words: 0.75. d1 and d2 "the package has not arrived" share none: 0, although they mean the same.

## 4. TF-IDF by hand

scikit-learn's `TfidfVectorizer` uses, by default:

- tf = the raw count of the word in the document;
- idf = ln((1 + N) / (1 + df)) + 1, with N documents and df the number of documents that contain the word;
- weight = tf × idf, and then every row divided by its length (`norm='l2'`).

> **Your turn.** Write `tfidf_by_hand(C)` that returns this matrix for a count matrix `C`. Use `(C > 0).sum(axis=0)` for df, `np.log` for ln, and `np.linalg.norm(W, axis=1, keepdims=True)` for the row lengths.

In [ ]:
def tfidf_by_hand(C):
    ...

W = tfidf_by_hand(C)

In [ ]:
expect_hash('your TF-IDF matrix (4 decimals)', np.round(W, 4) if isinstance(W, np.ndarray) else W, '9a5e1e2bf835f0f1')

> **Your turn.** The lesson's guided practice: the weight of "card" in d3 "refund my card". Read it from your matrix into `card_d3` (row 2, column `words.index("card")`). Then check it by hand: idf of card = ln(6/2) + 1, and the row length of d3.

In [ ]:
card_d3 = ...

In [ ]:
expect('card_d3', card_d3, 0.6902, tolerance=0.0001)

## 5. Compare with scikit-learn

Run the cell. It fits `TfidfVectorizer` with its defaults and compares every number with yours.

In [ ]:
sk = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
S = sk.fit_transform(docs).toarray()
print("idf:", sk.idf_.round(4))
if isinstance(W, np.ndarray) and W.shape == S.shape:
    print("largest difference to your matrix:", np.abs(S - W).max())
    print("equal to 4 decimals:", np.array_equal(S.round(4), W.round(4)))
else:
    print("Your tfidf_by_hand() has no result yet: complete section 4, then run this cell again.")

> **Check.** The largest difference is about 1e-16, the rounding error of a float64 number, and the last line prints `True`. Your six lines compute what the library computes.

## 6. Which option changes what?

`TfidfVectorizer` has two switches: `smooth_idf=False` uses idf = ln(N / df) + 1, and `sublinear_tf=True` uses 1 + ln(tf) instead of tf.

> **Predict.** On these five documents, one switch changes the weight of "parcel" in d1 and the other changes nothing. Which is which?

Run the cell.

In [ ]:
for options in [{"smooth_idf": False}, {"sublinear_tf": True}]:
    M = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, **options).fit_transform(docs).toarray()
    print(options, "| parcel in d1:", M[0, words.index("parcel")].round(4), "| largest change:", np.abs(M - S).max().round(4))

> **Check.** `smooth_idf=False`: "parcel" in d1 becomes 0.5255, and the largest change is 0.0402. `sublinear_tf=True` changes nothing: no word appears twice in these documents, and 1 + ln 1 = 1.

The textbook idf, ln(N / df) without any "+ 1", is a third variant again. Always write down which formula a number comes from.

## 7. On the tickets: idf, and two routers

Fit TF-IDF on the training tickets, look at the lowest and the highest idf, and train logistic regression on counts and on TF-IDF. It takes a few seconds.

> **Predict.** Which representation routes more validation tickets right: raw counts or TF-IDF? By how many tickets of 600?

In [ ]:
from sklearn.linear_model import LogisticRegression

texts, v_texts = [r["text"] for r in train_rows], [r["text"] for r in valid_rows]
y_train, y_valid = [r["team"] for r in train_rows], [r["team"] for r in valid_rows]
counts = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, min_df=2).fit(texts)
tfidf = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, min_df=2).fit(texts)

names, idf_t = tfidf.get_feature_names_out(), tfidf.idf_
order = idf_t.argsort()
print("lowest idf: ", [(names[i], round(float(idf_t[i]), 2)) for i in order[:5]])
print("highest idf:", [(names[i], round(float(idf_t[i]), 2)) for i in order[-3:]])

accuracy = {}
for name, vec in [("counts", counts), ("TF-IDF", tfidf)]:
    model = LogisticRegression(max_iter=3000).fit(vec.transform(texts), y_train)
    accuracy[name] = model.score(vec.transform(v_texts), y_valid)
    print(f"{name:7} validation accuracy {accuracy[name]:.4f}")

In [ ]:
expect('TF-IDF accuracy', round(accuracy['TF-IDF'], 4), 0.9883, tolerance=0.004)

> **Check.** Counts 0.9783, TF-IDF 0.9883: 6 more right tickets of 600. The full stop and "the" get the lowest idf; words in only two tickets get the highest, 8.36. Another computer can give a slightly different accuracy.

## 8. The synonym problem in three representations

Counts and TF-IDF give d1 and d2 a cosine of 0, because they share no word. Embeddings give every word a dense vector learned from text. The next cell downloads a small subset of the **GloVe** word vectors (50 numbers per word, 20,367 words, 3.9 MB; Pennington, Socher and Manning, 2014, public domain under the ODC PDDL 1.0) and checks its checksum.

Run the cell.

In [ ]:
import hashlib
glove_path = Path("data/glove6b-50d-subset.npz")
GLOVE_SHA = "da7efd39587326e233c16ffa8a8a89802f11de507bebaef1bb5b857d1f895f30"
if not glove_path.exists() or hashlib.sha256(glove_path.read_bytes()).hexdigest() != GLOVE_SHA:
    download(LABS + "data-glove/glove6b-50d-subset.npz", glove_path)
    if hashlib.sha256(glove_path.read_bytes()).hexdigest() != GLOVE_SHA:
        raise SystemExit("The GloVe file has the wrong checksum. Delete data/glove6b-50d-subset.npz and run again.")
glove = np.load(glove_path)
G = glove["vectors"]
row_of = {w: i for i, w in enumerate(glove["words"].tolist())}
print(G.shape, "| 'parcel' starts", G[row_of["parcel"]][:4].round(3))

> **Your turn.** With one-hot vectors, the cosine of "parcel" and "package" is 0. Compute their cosine with the GloVe vectors in `parcel_package`, with your `cosine()` from section 3.

In [ ]:
parcel_package = ...

In [ ]:
expect('parcel_package', parcel_package, 0.39, tolerance=0.005)

Now turn each document into **one** vector: the average of its words' GloVe vectors. Then compare d1 with d2 (same meaning, other words) and with d5 (shared words, other team) in all three representations.

> **Predict.** With averaged GloVe vectors, will d2 finally be closer to d1 than d5 is?

Run the cell.

In [ ]:
def average_vector(doc):
    return np.mean([G[row_of[t]] for t in tokenize(doc) if t in row_of], axis=0)

def cos(x, y):
    """Your cosine() from section 3, or the reference formula until it returns a number."""
    value = cosine(x, y)
    if isinstance(value, (int, float, np.floating)):
        return float(value)
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

if not isinstance(cosine(C[0], C[4]), (int, float, np.floating)):
    print("(Your cosine() from section 3 has no result yet: this cell uses the reference formula.)")
A = np.stack([average_vector(d) for d in docs])
for name, M in [("counts", C), ("TF-IDF", S), ("GloVe average", A)]:
    print(f"{name:14} d1-d2 {cos(M[0], M[1]):.3f}   d1-d5 {cos(M[0], M[4]):.3f}")

> **Check.** Counts 0.000 and 0.750, TF-IDF 0.000 and 0.729. The GloVe average gives d1-d2 0.905 and d1-d5 0.939: d2 is no longer at 0, but **every** pair now looks alike. "my", "is" and "the" pull every average to the same place. A plain average of general word vectors is a weak sentence vector.

The lesson uses a trained sentence-embedding model instead, **all-MiniLM-L6-v2** (Apache-2.0, about 90 MB). The next cell is **optional**: it runs only if the package `sentence-transformers` is installed (Colab has it). On another computer, install it with `%pip install -q sentence-transformers` first. It downloads the model the first time.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None
    print("sentence-transformers is not installed. This step is optional: the lesson shows its output.")

if SentenceTransformer is not None:
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    E = encoder.encode(docs, normalize_embeddings=True)
    print(E.shape, "| d1-d2", round(float(E[0] @ E[1]), 3), "| d1-d5", round(float(E[0] @ E[4]), 3))

> **Check.** (5, 384), d1-d2 0.502 and d1-d5 0.573: the trained sentence model brings the two ways to say "my parcel is late" close together. On routing, though, frozen MiniLM vectors reach only 94.50% of the validation tickets against 98.83% for TF-IDF (the lesson's chart): there, the exact words decide.

## Next

You computed counts, cosine similarity and TF-IDF by hand, matched scikit-learn to 4 decimals, and measured the synonym problem in three representations. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m08/from-counts-to-meaning/) has the explorer, the charts and the knowledge checks. The next lesson shows how word vectors such as GloVe are learned.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [How embeddings learn meaning: word2vec and GloVe](https://learning.nextia-ai.com/courses/deep-learning/m08/word2vec-and-glove/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m08/from-counts-to-meaning/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.